In [1]:
from datasets import load_dataset
def load_data():
    dataset = load_dataset("stanfordnlp/imdb")
    return dataset

In [2]:
# Step 1 : Load the dataset
dataset = load_data()
dataset

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

In [3]:
def split_data(dataset):
    train_valid = dataset['train'].train_test_split(test_size=0.1 ,seed=42)
    
    train_dataset = train_valid['train']
    valid_dataset = train_valid['test']
    test_dataset  = dataset['test']
    
    return train_dataset, valid_dataset, test_dataset

In [4]:
# Step 2 : Split the dataset into training, validation, and test sets
train_dataset, valid_dataset, test_dataset = split_data(dataset)

In [5]:
from transformers import BertTokenizer, BertForSequenceClassification

model_name = 'bert-base-uncased'

tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertForSequenceClassification.from_pretrained(model_name, num_labels=2)

print(f"Tokenizer loaded: {type(tokenizer).__name__}")
print(f"Model loaded: {type(model).__name__}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Tokenizer loaded: BertTokenizer
Model loaded: BertForSequenceClassification


In [ ]:
from transformers import BertTokenizer

def tokenize_data(train_dataset, valid_dataset, test_dataset):
    
    
    tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
    
    def tokenize(example):
        return tokenizer(
            example['text'],
            padding='max_length',
            truncation=True,
            max_length=384
        )
    
    train_dataset = train_dataset.map(tokenize, batched=True)
    valid_dataset = valid_dataset.map(tokenize, batched=True)
    test_dataset  = test_dataset.map(tokenize, batched=True)
    
    return train_dataset, valid_dataset, test_dataset, tokenizer

In [7]:
# Step 3 : Tokenize the datasets
train_dataset, valid_dataset, test_dataset, tokenizer = tokenize_data(
        train_dataset, valid_dataset, test_dataset
)

Map:   0%|          | 0/2500 [00:00<?, ? examples/s]

In [8]:
from transformers import BertForSequenceClassification
def load_model():
    
    model = BertForSequenceClassification.from_pretrained(
        'bert-base-uncased',
        num_labels=2
    )
    
    return model

In [9]:
# Step 4 : Load the pre-trained BERT model for sequence classification
model = load_model()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [10]:
def freeze_bert(model):
    for param in model.bert.parameters():
        param.requires_grad = False
    return model


def unfreeze_bert(model):
    for param in model.bert.parameters():
        param.requires_grad = True
    return model

In [11]:
def get_training_args(output_dir='./results', epochs=4,lr=2e-5):
    from transformers import TrainingArguments
    
    training_args = TrainingArguments(
        output_dir=output_dir,
        do_eval=True,
        learning_rate=lr,
        per_device_train_batch_size=16,
        num_train_epochs=epochs,
        weight_decay=0.01
    )
    
    return training_args

In [12]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):
    
    logits, labels = eval_pred
    preds = logits.argmax(axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average='binary' , zero_division=0
    )
    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall
    }

In [13]:
from transformers import Trainer

def create_trainer(model, training_args, train_dataset, valid_dataset):
    
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=valid_dataset,
        compute_metrics=compute_metrics
    )
    
    return trainer

In [14]:
def train_model(trainer):
    trainer.train()
    return trainer

In [15]:
# Stage 1 : Freeze BERT layers and train only the classification head
model = freeze_bert(model)
training_args = get_training_args(lr=1e-3)
trainer = create_trainer(model, training_args, train_dataset, valid_dataset)
trainer = train_model(trainer)

Step,Training Loss
500,0.608985
1000,0.536668
1500,0.513239
2000,0.504044
2500,0.495783
3000,0.492171
3500,0.493531
4000,0.479212
4500,0.480046
5000,0.480304


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [16]:
# Stage 2 : Unfreeze BERT layers and fine-tune the entire model
model = unfreeze_bert(model)
training_args = get_training_args(lr=2e-5)
trainer = create_trainer(model, training_args, train_dataset, valid_dataset)
trainer = train_model(trainer)

Step,Training Loss
500,0.292963
1000,0.234732
1500,0.216257
2000,0.150767
2500,0.145873
3000,0.104897
3500,0.069781
4000,0.055024
4500,0.045634
5000,0.026272


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [17]:
def evaluate_model(trainer, test_dataset):
    results = trainer.evaluate(test_dataset)
    return results

In [18]:
# Evaluation 
results = evaluate_model(trainer, test_dataset)
print(results)

Training Loss,Validation Loss,Step,Accuracy,F1,Precision,Recall
0.023155,0.430120,5628,0.934200,0.934454,0.930857,0.938080


{'eval_loss': 0.4301195442676544, 'eval_accuracy': 0.9342, 'eval_f1': 0.9344543172490736, 'eval_precision': 0.9308565531475749, 'eval_recall': 0.93808}


In [19]:
def analyze_performance(trainer, test_dataset):
    import time
    
    # Time measurement
    start = time.time()
    trainer.evaluate(test_dataset)
    end = time.time()
    
    print("Evaluation Time:", end - start)
    
    # Predictions
    preds = trainer.predict(test_dataset)
    
    return preds

In [20]:
# Analysis
analyze_performance(trainer, test_dataset)

Training Loss,Validation Loss,Step,Accuracy,F1,Precision,Recall
0.023155,0.430120,5628,0.934200,0.934454,0.930857,0.938080


Evaluation Time: 652.2004866600037


PredictionOutput(predictions=array([[ 5.370338 , -5.234225 ],
       [ 4.128406 , -3.9732065],
       [ 4.940412 , -4.740572 ],
       ...,
       [-3.997642 ,  4.0972233],
       [ 2.7155669, -2.7026699],
       [-4.6612773,  4.7313094]], dtype=float32), label_ids=array([0, 0, 0, ..., 1, 1, 1]), metrics={'test_loss': 0.4301195442676544, 'test_accuracy': 0.9342, 'test_f1': 0.9344543172490736, 'test_precision': 0.9308565531475749, 'test_recall': 0.93808, 'test_runtime': 873.8274, 'test_samples_per_second': 28.61, 'test_steps_per_second': 3.576})

In [21]:
def save_model(trainer, tokenizer, save_path="./bert_model"):
    # Save model
    trainer.save_model(save_path)
    
    # Save tokenizer
    tokenizer.save_pretrained(save_path)
    
    print(f"Model saved at {save_path}")

In [22]:
# Save
save_model(trainer, tokenizer)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved at ./bert_model


In [23]:
from transformers import BertForSequenceClassification, BertTokenizer

def load_saved_model(save_path="./bert_model"):
    
    model = BertForSequenceClassification.from_pretrained(save_path)
    tokenizer = BertTokenizer.from_pretrained(save_path)
    
    return model, tokenizer

In [24]:
import torch

def predict_sentiment(text, model, tokenizer):
    model.eval()
    
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )
    
    # Move to same device (important if using GPU)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model(**inputs)
    
    logits = outputs.logits
    
    # Softmax for probabilities
    probs = torch.softmax(logits, dim=1)
    
    prediction = torch.argmax(probs, dim=1).item()
    confidence = probs[0][prediction].item()
    
    label = "Positive" if prediction == 1 else "Negative"
    
    return label, confidence

In [25]:
model, tokenizer = load_saved_model()

text = "An emotionally resonant experience with nuanced performances that stay with you long after the credits roll. The director has crafted a beautiful story that is both intimate and deeply moving. Highly recommended."

result = predict_sentiment(text, model, tokenizer)

print(result)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

('Positive', 0.9993869066238403)


In [26]:
model, tokenizer = load_saved_model()

text = "I grew up...watching and loving [the original]. What they did to this is a disaster...completely ruined the legacy"

result = predict_sentiment(text, model, tokenizer)

print(result)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

('Negative', 0.9992172718048096)
